# Construcción de un modelo de Machine Learning para imputación de datos hidrometeorológicos

**Alumno:** Luis Gustavo Cazarez Garcia  
**Materia:** Ciencia de Datos
**Actividad:** Construcción de un modelo de Machine Learning para imputación  

## Objetivo

Construir y validar un modelo de Machine Learning para imputar datos hidrometeorológicos.  
El dataset contiene variables de temperatura mínima, temperatura media, temperatura máxima y precipitación.

La variable evaporación fue solicitada en las instrucciones, pero al revisar el dataset proporcionado se identificó que no está incluida en el archivo CSV.

In [1]:
import math
import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 1. Carga del dataset

En esta sección se carga el archivo `data.csv` y se revisan sus dimensiones, columnas y primeras filas.

In [2]:
df = pd.read_csv("data.csv")

df["PERIODO"] = pd.to_datetime(df["PERIODO"], errors="coerce")

print("Dimensiones del dataset:")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

df.head()

Dimensiones del dataset:
(16368, 7)

Columnas del dataset:
['PERIODO', 'CVE_ENT', 'ENTIDAD', 'MINIMA', 'MEDIA', 'MAXIMA', 'PRECIPITACION']


,PERIODO,CVE_ENT,ENTIDAD,MINIMA,MEDIA,MAXIMA,PRECIPITACION
0,1985-01-01,0,Nacional,7.8,15.9,23.9,36.0
1,1985-01-01,1,Aguascalientes,3.1,12.2,21.3,4.9
2,1985-01-01,2,Baja California,5.6,12.9,20.2,12.2
3,1985-01-01,3,Baja California Sur,9.2,17.1,25.0,30.3
4,1985-01-01,4,Campeche,15.7,22.7,29.7,20.9


## 2. Análisis de valores faltantes

Se revisan los valores faltantes en las variables principales: temperatura mínima, temperatura media, temperatura máxima y precipitación.  
También se revisa evaporación, aunque esta variable no aparece en el dataset proporcionado.

In [3]:
variables_hidro = ["MINIMA", "MEDIA", "MAXIMA", "PRECIPITACION"]
variables_revisar = variables_hidro + ["EVAPORACION"]

resultados_faltantes = []

for variable in variables_revisar:
    if variable in df.columns:
        faltantes = df[variable].isna().sum()
        porcentaje = df[variable].isna().mean() * 100
        
        resultados_faltantes.append({
            "Variable": variable,
            "Valores faltantes": faltantes,
            "Porcentaje faltante": round(porcentaje, 2),
            "Estado": "Disponible en el dataset"
        })
    else:
        resultados_faltantes.append({
            "Variable": variable,
            "Valores faltantes": "No aplica",
            "Porcentaje faltante": "No aplica",
            "Estado": "No existe en el dataset"
        })

tabla_faltantes = pd.DataFrame(resultados_faltantes)
tabla_faltantes

,Variable,Valores faltantes,Porcentaje faltante,Estado
0,MINIMA,0,0.0,Disponible en el dataset
1,MEDIA,0,0.0,Disponible en el dataset
2,MAXIMA,0,0.0,Disponible en el dataset
3,PRECIPITACION,0,0.0,Disponible en el dataset
4,EVAPORACION,No aplica,No aplica,No existe en el dataset


## 3. Interpretación de los valores faltantes

El análisis muestra que el dataset no presenta valores faltantes reales en las variables disponibles: MINIMA, MEDIA, MAXIMA y PRECIPITACION.

También se identificó que la variable EVAPORACION no existe dentro del dataset proporcionado. Por esta razón, no se puede imputar dicha variable y se considera una limitación del análisis.

Como no existen valores faltantes reales, se aplicará una técnica de validación por enmascaramiento artificial. Esta técnica consiste en ocultar temporalmente una parte de los datos conocidos para evaluar si el modelo puede recuperarlos correctamente.

## 4. Metodología seleccionada

La metodología seleccionada para este proyecto fue CRISP-DM, debido a que organiza el proceso de análisis de datos y Machine Learning en etapas claras:

1. Comprensión del problema.
2. Comprensión de los datos.
3. Preparación de los datos.
4. Modelado.
5. Evaluación.
6. Aplicación del modelo.

Esta metodología es adecuada porque permite documentar cada decisión tomada durante el proceso de imputación. Además, evita realizar reemplazos automáticos sin analizar primero la estructura y calidad del dataset.

Para el modelo se utilizó IterativeImputer con `RandomForestRegressor`. Random Forest es adecuado para datos hidrometeorológicos porque puede aprender relaciones no lineales entre variables como temperatura mínima, media, máxima y precipitación.

## 5. Preparación de variables

Se preparan las variables que serán utilizadas por el modelo.  
A partir de la columna PERIODO se generan variables temporales como año y mes.

También se agregan transformaciones cíclicas del mes usando seno y coseno, ya que las variables climáticas pueden presentar patrones estacionales.

In [4]:
df_modelo = df.copy()

df_modelo["ANIO"] = df_modelo["PERIODO"].dt.year
df_modelo["MES"] = df_modelo["PERIODO"].dt.month

df_modelo["MES_SIN"] = np.sin(2 * np.pi * df_modelo["MES"] / 12)
df_modelo["MES_COS"] = np.cos(2 * np.pi * df_modelo["MES"] / 12)

columnas_modelo = [
    "CVE_ENT",
    "ANIO",
    "MES",
    "MES_SIN",
    "MES_COS",
    "MINIMA",
    "MEDIA",
    "MAXIMA",
    "PRECIPITACION"
]

df_modelo = df_modelo[columnas_modelo].copy()

df_modelo.head()

,CVE_ENT,ANIO,MES,MES_SIN,MES_COS,MINIMA,MEDIA,MAXIMA,PRECIPITACION
0,0,1985,1,0.5,0.866025,7.8,15.9,23.9,36.0
1,1,1985,1,0.5,0.866025,3.1,12.2,21.3,4.9
2,2,1985,1,0.5,0.866025,5.6,12.9,20.2,12.2
3,3,1985,1,0.5,0.866025,9.2,17.1,25.0,30.3
4,4,1985,1,0.5,0.866025,15.7,22.7,29.7,20.9


## 6. Construcción del modelo

El modelo utilizado fue una imputación iterativa con Random Forest.

La imputación iterativa estima una variable utilizando las demás variables disponibles como predictoras.  
Random Forest trabaja con varios árboles de decisión y combina sus resultados para obtener una predicción más estable.

Los hiperparámetros utilizados fueron:

- `n_estimators = 120`
- `random_state = 42`
- `min_samples_leaf = 2`
- `max_features = sqrt`
- `max_iter = 5`
- `initial_strategy = median`

Estos valores permiten tener un modelo estable, reproducible y con menor riesgo de sobreajuste.

In [6]:
def crear_imputador():
    modelo_rf = RandomForestRegressor(
        n_estimators=120,
        random_state=42,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1
    )

    imputador = IterativeImputer(
        estimator=modelo_rf,
        max_iter=5,
        initial_strategy="median",
        random_state=42,
        tol=1e-3,
        skip_complete=True
    )

    return imputador

## 7. Validación del modelo

Como el dataset no tiene valores faltantes reales, se aplicó validación por enmascaramiento artificial.

El procedimiento fue:

1. Seleccionar una variable.
2. Ocultar artificialmente el 20% de sus valores conocidos.
3. Usar el modelo para imputar esos valores.
4. Comparar los valores imputados contra los valores reales originales.
5. Calcular métricas de regresión: MSE, RMSE, MAE y R².

In [7]:
rng = np.random.default_rng(42)
resultados = []

for variable_objetivo in variables_hidro:
    datos_validacion = df_modelo.copy()

    indices_observados = datos_validacion.index[
        datos_validacion[variable_objetivo].notna()
    ].to_numpy()

    cantidad_prueba = int(len(indices_observados) * 0.20)

    indices_prueba = rng.choice(
        indices_observados,
        size=cantidad_prueba,
        replace=False
    )

    valores_reales = datos_validacion.loc[indices_prueba, variable_objetivo].copy()

    datos_validacion.loc[indices_prueba, variable_objetivo] = np.nan

    imputador = crear_imputador()
    datos_imputados = imputador.fit_transform(datos_validacion)

    datos_imputados = pd.DataFrame(
        datos_imputados,
        columns=datos_validacion.columns,
        index=datos_validacion.index
    )

    valores_predichos = datos_imputados.loc[indices_prueba, variable_objetivo]

    mse = mean_squared_error(valores_reales, valores_predichos)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(valores_reales, valores_predichos)
    r2 = r2_score(valores_reales, valores_predichos)

    resultados.append({
        "Variable": variable_objetivo,
        "n_prueba": cantidad_prueba,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

resultados_validacion = pd.DataFrame(resultados)
resultados_validacion

,Variable,n_prueba,MSE,RMSE,MAE,R2
0,MINIMA,3273,0.248224,0.498220,0.362494,0.992212
1,MEDIA,3273,0.028520,0.168878,0.107324,0.998755
2,MAXIMA,3273,0.234240,0.483983,0.357856,0.986998
3,PRECIPITACION,3273,2292.048785,47.875346,27.795832,0.740274


## 8. Interpretación de resultados

Los resultados muestran que el modelo tuvo un desempeño excelente en las variables de temperatura.

La variable MEDIA obtuvo el mejor resultado, con un R² aproximado de 0.9987. Esto significa que el modelo logró explicar casi toda la variabilidad de los datos ocultos artificialmente.

Las variables MINIMA y MAXIMA también obtuvieron resultados muy altos, con R² superiores a 0.98. Esto indica que el modelo aprendió correctamente la relación entre las variables de temperatura y las variables temporales.

La variable PRECIPITACION obtuvo un R² aproximado de 0.7402. Este desempeño es menor que el de las temperaturas, pero sigue siendo aceptable. La precipitación es más difícil de estimar porque suele ser irregular, con cambios bruscos y eventos extremos.

## 9. Generación del dataset imputado

Aunque el dataset no tiene valores faltantes reales, se genera un archivo final con indicadores de imputación.

Estos indicadores permiten saber si un valor fue imputado o si ya existía originalmente en el dataset.

In [8]:
imputador_final = crear_imputador()
datos_imputados_final = imputador_final.fit_transform(df_modelo)

datos_imputados_final = pd.DataFrame(
    datos_imputados_final,
    columns=df_modelo.columns,
    index=df_modelo.index
)

df_final = df.copy()

for columna in variables_hidro:
    mascara_faltantes = df_final[columna].isna()
    df_final[columna + "_fue_imputado"] = mascara_faltantes

    df_final.loc[mascara_faltantes, columna] = datos_imputados_final.loc[
        mascara_faltantes,
        columna
    ]

df_final["PRECIPITACION"] = df_final["PRECIPITACION"].clip(lower=0)

df_final.head()

,PERIODO,CVE_ENT,ENTIDAD,MINIMA,MEDIA,MAXIMA,PRECIPITACION,MINIMA_fue_imputado,MEDIA_fue_imputado,MAXIMA_fue_imputado,PRECIPITACION_fue_imputado
0,1985-01-01,0,Nacional,7.8,15.9,23.9,36.0,False,False,False,False
1,1985-01-01,1,Aguascalientes,3.1,12.2,21.3,4.9,False,False,False,False
2,1985-01-01,2,Baja California,5.6,12.9,20.2,12.2,False,False,False,False
3,1985-01-01,3,Baja California Sur,9.2,17.1,25.0,30.3,False,False,False,False
4,1985-01-01,4,Campeche,15.7,22.7,29.7,20.9,False,False,False,False


In [9]:
df_final.to_csv("data_imputado_con_indicadores.csv", index=False)
resultados_validacion.to_csv("resultados_validacion.csv", index=False)
tabla_faltantes.to_csv("analisis_faltantes.csv", index=False)

print("Archivos generados correctamente:")
print("- data_imputado_con_indicadores.csv")
print("- resultados_validacion.csv")
print("- analisis_faltantes.csv")

Archivos generados correctamente:
- data_imputado_con_indicadores.csv
- resultados_validacion.csv
- analisis_faltantes.csv


## 10. Conclusión

En este proyecto se construyó un modelo de Machine Learning para imputación de datos hidrometeorológicos. El dataset analizado contiene temperatura mínima, temperatura media, temperatura máxima y precipitación.

Durante el análisis inicial se encontró que no existían valores faltantes reales en las variables disponibles. También se identificó que la variable evaporación no estaba incluida en el dataset. Por ello, la validación se realizó mediante enmascaramiento artificial, ocultando el 20% de los datos conocidos.

El modelo utilizado fue IterativeImputer con Random Forest Regressor. Los resultados fueron excelentes para las variables de temperatura, con valores de R² superiores a 0.98. En precipitación el desempeño fue menor, con un R² aproximado de 0.74, pero sigue siendo aceptable debido a la naturaleza irregular de la lluvia.

El proyecto demuestra que Machine Learning puede utilizarse para imputar datos hidrometeorológicos, siempre que se analicen primero las características del dataset, se justifique la metodología y se validen los resultados con métricas apropiadas.